In [1]:
import stlearn as st
import pandas as pd
import pathlib as pathlib
import matplotlib.pyplot as plt

st.settings.set_figure_params(dpi=120)

# Ignore all warnings
import warnings
warnings.filterwarnings("ignore")

/home/xx244/.conda/envs/software/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/home/xx244/.conda/envs/software/lib/python3.10/site-packages/stlearn/tl/cci/het.py:206: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit(parallel=True, nopython=False)


In [2]:
import torch
import anndata as ad
import numpy as np

def get_int_df(adata, use_label, lr=None, sig_interactions=True, title=None):
    """Retrieves the relevant interaction count matrix."""
    no_title = title is None
    labels_ordered = adata.obs[use_label].cat.categories
    if lr is None:  # No LR inputted, so just use all
        int_df = (
            adata.uns[f"lr_cci_{use_label}"]
            if sig_interactions
            else adata.uns[f"lr_cci_raw_{use_label}"]
        )[labels_ordered].loc[labels_ordered]
        title = "Cell-Cell LR Interactions" if no_title else title
    else:
        labels_ordered = adata.obs[use_label].cat.categories
        int_df = (
            adata.uns[f"per_lr_cci_{use_label}"][lr]
            if sig_interactions
            else adata.uns[f"per_lr_cci_raw_{use_label}"][lr]
        )[labels_ordered].loc[labels_ordered]

        title = f"Cell-Cell {lr} interactions" if no_title else title

    return int_df, title

# Mouse brain

In [5]:


df=pd.read_csv("./data/mouse/mouse.csv")
df=df[df['slice_id']=="mouse1_slice201"].copy()
print(df.shape)
genes=torch.load("./data/mouse/genes.pth")
adata=ad.AnnData(X=df[genes].values)
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.var_names=genes

adata.obs['imagecol']=df["centerx"].values
adata.obs['imagerow']=df["centery"].values
adata.obs["louvain"]=df["subclass"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 10/4.71, 'spot_diameter_fullres': 50*10/4.71}}}

(6137, 275)


In [6]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

PCA is done! Generated in adata.obsm['X_pca'], adata.uns['pca'] and adata.varm['PCs']
Created k-Nearest-Neighbor graph in adata.uns['neighbors'] 
Normalization step is finished in adata.X


In [7]:
### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.

125 by 125 has this many spots:
 15625
Gridding...
(4787, 254)


In [8]:
# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='mouse')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

2293
Calculating neighbours...
0 spots with no neighbours, 32 median spot neighbours.
Spot neighbour indices stored in adata.obsm['spot_neighbours'] & adata.obsm['spot_neigh_bcs'].
Altogether 6 valid L-R pairs


Generating backgrounds & testing each LR pair...: 100%|██████████ [ time left: 00:00 ]



Storing results:

lr_scores stored in adata.obsm['lr_scores'].
p_vals stored in adata.obsm['p_vals'].
p_adjs stored in adata.obsm['p_adjs'].
-log10(p_adjs) stored in adata.obsm['-log10(p_adjs)'].
lr_sig_scores stored in adata.obsm['lr_sig_scores'].

Per-spot results in adata.obsm have columns in same order as rows in adata.uns['lr_summary'].
Summary of LR results in adata.uns['lr_summary'].
(6, 3)
              n_spots  n_spots_sig  n_spots_sig_pval
Ptprm_Ptprm      3032          502               860
Ptprk_Ptprk      3432          286               962
Vtn_Itgb8        3848          125               236
Vip_Vipr2        4318          122               234
Pdgfc_Pdgfra     2585          114               237
Prok2_Prokr2      670           26                72
Updated adata.uns[lr_summary]
Updated adata.obsm[lr_scores]
Updated adata.obsm[lr_sig_scores]
Updated adata.obsm[p_vals]
Updated adata.obsm[p_adjs]
Updated adata.obsm[-log10(p_adjs)]
Getting cached neighbourhood information...


Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Significant counts of cci_rank interactions for all LR pairs in data.uns[lr_cci_louvain]
Significant counts of cci_rank interactions for each LR pair stored in dictionary data.uns[per_lr_cci_louvain]


In [11]:
int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/mouse.csv")

            Astro  Endo  L2/3 IT  L4/5 IT  L5 ET  L5 IT  L5/6 NP  L6 CT  \
Astro          39     0       46       65      0      0        0      0   
Endo          444   455      708     1378    190    864       56    394   
L2/3 IT         0     0      662        0      0      0        0      0   
L4/5 IT       137   508        0     2981      0    479        0      0   
L5 ET           0     0        0        2      8      0        1      0   
L5 IT           0   205        0      241    110    460       40      0   
L5/6 NP         0     0        0        0      0      0        3      0   
L6 CT           0     0        0        0      0      0        0      7   
L6 IT           0     0        0        0      0      0        0      0   
L6 IT Car3      0     0        0        0      0      0        0      0   
L6b             0     0        0        0      0      0        0     16   
Lamp5          56    88      120       67      0     35        0      0   
Micro          11    97  

# AD

In [10]:
df=pd.read_csv("./data/AD/AD.csv")
df=df[df["section"]=="H20.33.001.CX28.MTG.02.007.1.02.03"].copy()
genes=torch.load("./data/AD/genes.pth")
adata=ad.AnnData(X=df[genes].values)
adata.obs["centerx"]=df["centerx"].values
adata.obs["centery"]=df["centery"].values

adata.var_names=genes
print(adata)

adata.obs['imagecol']=df["centerx"].values
adata.obs['imagerow']=df["centery"].values
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.obs["louvain"]=df["subclass"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 10/4.71, 'spot_diameter_fullres': 50*10/4.71}}}

AnnData object with n_obs × n_vars = 15225 × 140
    obs: 'centerx', 'centery'


In [12]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.


# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/AD.csv")

PCA is done! Generated in adata.obsm['X_pca'], adata.uns['pca'] and adata.varm['PCs']
Created k-Nearest-Neighbor graph in adata.uns['neighbors'] 
Normalization step is finished in adata.X
125 by 125 has this many spots:
 15625
Gridding...
(6641, 140)
2293
Calculating neighbours...
0 spots with no neighbours, 50 median spot neighbours.
Spot neighbour indices stored in adata.obsm['spot_neighbours'] & adata.obsm['spot_neigh_bcs'].
Altogether 5 valid L-R pairs


Generating backgrounds & testing each LR pair...: 100%|██████████ [ time left: 00:00 ]



Storing results:

lr_scores stored in adata.obsm['lr_scores'].
p_vals stored in adata.obsm['p_vals'].
p_adjs stored in adata.obsm['p_adjs'].
-log10(p_adjs) stored in adata.obsm['-log10(p_adjs)'].
lr_sig_scores stored in adata.obsm['lr_sig_scores'].

Per-spot results in adata.obsm have columns in same order as rows in adata.uns['lr_summary'].
Summary of LR results in adata.uns['lr_summary'].
(5, 3)
             n_spots  n_spots_sig  n_spots_sig_pval
ROBO1_ROBO1     4516          288               625
ROBO2_ROBO2     3187          196               494
DCN_EGFR        3921          123               200
SLIT3_ROBO1     5379           94               161
SLIT3_ROBO2     4599           93               178
Updated adata.uns[lr_summary]
Updated adata.obsm[lr_scores]
Updated adata.obsm[lr_sig_scores]
Updated adata.obsm[p_vals]
Updated adata.obsm[p_adjs]
Updated adata.obsm[-log10(p_adjs)]
Getting cached neighbourhood information...
Getting information for CCI counting...


Counting celltype-celltype interactions per LR and permuting 100 times.: 100%|██████████ [ time left: 00:00 ]

Significant counts of cci_rank interactions for all LR pairs in data.uns[lr_cci_louvain]
Significant counts of cci_rank interactions for each LR pair stored in dictionary data.uns[per_lr_cci_louvain]
                 Astrocyte  Chandelier  Endothelial  L2/3 IT  L4 IT  L5 ET  \
Astrocyte                0           0            0        0      0      0   
Chandelier               0           0            0        0      0      0   
Endothelial            307           0          182      356    130     11   
L2/3 IT                  0           0           63      681      0      0   
L4 IT                    0           0            0        0     99      0   
L5 ET                    0           0            0        0      0      3   
L5 IT                    0           0            0        0      0     28   
L5/6 NP                  0           0            0        0      0      0   
L6 CT                  107           0           58        0      0      0   
L6 IT               

# NSCLC

In [ ]:
df=pd.read_csv("./data/NSCLC/NSCLC.csv")
df=df[df["section"]=="Lung6"].copy()
print(df.columns)
genes=torch.load("./data/NSCLC/genes.pth")
adata=ad.AnnData(X=df[genes].values)
adata.obs["centerx"]=df['CenterX_global_px'].values
adata.obs["centery"]=df['CenterY_global_px'].values
adata.obsm["spatial"]=np.stack([df['CenterX_global_px'].values,df['CenterY_global_px'].values],axis=-1)
adata.var_names=genes
print(adata)

adata.obs['imagecol']=df['CenterX_global_px'].values
adata.obs['imagerow']=df['CenterY_global_px'].values
adata.obsm["spatial"]=np.stack([df['CenterX_global_px'].values,df['CenterY_global_px'].values],axis=-1)
adata.obs["louvain"]=df["CellType"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 8.32/4.71, 'spot_diameter_fullres': 50*8.32/4.71}}}

In [ ]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.


# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/NSCLC.csv")

# BC Xenium

In [ ]:
df=pd.read_csv("./data/BC/BC.csv")
df=df[df["section"]=="sample1_rep1"].copy()
print(df.columns)
genes=torch.load("./data/BC/genes.pth")

genes=[i for i in genes if i.find("_")<0]

print(genes)
adata=ad.AnnData(X=df[genes].values)
adata.obs["centerx"]=df["centerx"].values
adata.obs["centery"]=df["centery"].values
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.var_names=genes
print(adata)

adata.obs['imagecol']=df["centerx"].values
adata.obs['imagerow']=df["centery"].values
adata.obsm["spatial"]=np.stack([df["centerx"].values,df["centery"].values],axis=-1)
adata.obs["louvain"]=df["subclass"].values
adata.obs["louvain"]=adata.obs["louvain"].astype('category')
adata.uns['spatial']={'dataset':{'use_quality': 'hires', 'scalefactors': {'tissue_hires_scalef': 1, 'spot_diameter_fullres': 50}}}

In [ ]:
# QC - Filter genes and cells with at least 10 counts
st.pp.filter_genes(adata, min_counts=10)
st.pp.filter_cells(adata, min_counts=10)

# Store the raw data for using PSTS
adata.raw = adata
# Run PCA, neighbors and clustering.
st.em.run_pca(adata, n_comps=50, random_state=0)
st.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca', random_state=0)
#st.tl.clustering.louvain(adata, random_state=0)

#### Normalize total...
st.pp.normalize_total(adata)

### Calculating the number of grid spots we will generate
n_ = 125
print(f'{n_} by {n_} has this many spots:\n', n_ * n_)

### Gridding.
grid = st.tl.cci.grid(adata, n_row=n_, n_col=n_, use_label='louvain')
print(grid.shape)  # Slightly less than the above calculation, since we filter out spots with 0 cells.


# Loading the LR databases available within stlearn (from NATMI)
lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')#human!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
print(len(lrs))

# Running the analysis #
st.tl.cci.run(grid, lrs,
              min_spots=20,  # Filter out any LR pairs with no scores for less than min_spots
              distance=None,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
              n_pairs=1000,  # Number of random pairs to generate; low as example, recommend ~10,000
              n_cpus=None,   # Number of CPUs for parallel. If None, detects & use all available.
              )

lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
print(lr_info.shape)
print(lr_info)

### Can adjust significance thresholds.
st.tl.cci.adj_pvals(grid, correct_axis='spot',
                    pval_adj_cutoff=0.05, adj_method='fdr_bh')

best_lr = grid.uns['lr_summary'].index.values[0]  # Just choosing one of the top from lr_summary

st.tl.cci.run_cci(grid, 'louvain',  # Spot cell information either in data.obs or data.uns
                  min_spots=2,  # Minimum number of spots for LR to be tested.
                  spot_mixtures=True,  # If True will use the deconvolution data,
                  # so spots can have multiple cell types if score>cell_prop_cutoff
                  cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                  sig_spots=True,  # Only consider neighbourhoods of spots which had significant LR scores.
                  n_perms=100,  # Permutations of cell information to get background, recommend ~1000
                  n_cpus=None,
                  )

int_df, title = get_int_df(grid, "louvain")
print(int_df)

int_df.to_csv("./stLearn/BC.csv")